In [1]:
import numpy as np
import pandas as pd
import random

In [2]:
def best_gini(X, y,rng, k = 2):
    best_score, best_col, best_thresh = np.inf, None, None

    valid_cols = rng.choice([i for i in range(X.shape[1])], size=k, replace = False)
    for col in valid_cols:
        s = np.sort(np.unique(X[:, col]))
        thresholds = [(s[i] + s[i+1])/2 for i in range(len(s) - 1)]

        for thresh in thresholds:
            score = split_score(X, y, col, thresh)
            if score < best_score:
                best_score, best_col, best_thresh = score, col, thresh
                
    return (best_col, best_score, best_thresh)
        
def split_score(X, y, feature_column, threshold):
    true_array = y[X[:, feature_column] >= threshold]
    false_array = y[X[:, feature_column] < threshold]

    n1, n2 = len(true_array), len(false_array)
    true_pct, false_pct = n1/(n1+n2), n2/(n1+n2)
    score = true_pct * gini(true_array) + false_pct * gini(false_array)

    return score

def gini(labels):
    _, freq = np.unique(labels, return_counts = True)
    n = len(labels)
    return 1- sum([(f/n)**2 for f in freq])

#Returns the boostrapped and out-of-bag dataset
def boostrap_dataset(X, y, rng):
    n, _ = X.shape
    random_idxs = rng.choice([i for i in range(n)], size=n, replace = True)
    oob_idxs = np.setdiff1d(np.arange(n), random_idxs)
    return (X[random_idxs], y[random_idxs], X[oob_idxs], y[oob_idxs])

In [ ]:
from __future__ import annotations
from dataclasses import dataclass

@dataclass
class LeafNode:
    res: int

@dataclass
class DecisionNode:
    col: int
    threshold: float
    leftNode: DecisionNode | LeafNode | None = None
    rightNode: DecisionNode | LeafNode | None = None

In [ ]:
def _build(X, y, rng, depth = 2, score_cutoff = 0, k=2):   
    if not len(X) or not len(y):
        raise IndexError("Length was 0?")
    
    best_col, best_score, thresh = best_gini(X, y, rng, k=k)
    curr = DecisionNode(best_col, thresh)

    if depth == 0 or gini(y) - best_score <= score_cutoff:
        majority = np.argmax(np.bincount(y))
        return LeafNode(majority)
    else:

        left_x, left_y = X[X[:,best_col] >= thresh], y[X[:,best_col] >= thresh]
        right_x, right_y = X[X[:,best_col] < thresh], y[X[:,best_col] < thresh]
        curr.leftNode = _build(left_x, left_y, rng, depth-1, score_cutoff, k = k)
        curr.rightNode = _build(right_x, right_y, rng, depth-1, score_cutoff, k=k)
    return curr

def predict(X, tree):
    ret = []
    for sample in X:
        curr = tree
        while curr and type(curr) is DecisionNode:
            c = curr.col
            if sample[c] >= curr.threshold:
                curr = curr.leftNode
            else:
                curr = curr.rightNode
        ret.append(curr.res if curr else None)
    return ret

def create_random_forest(X, y, depth, num_trees = 3, k = 2, bootstrap = True, rng = None):
    ret = []
    for i in range(num_trees):
        if bootstrap:
            new_X, new_y, oob_X, oob_y = boostrap_dataset(X, y, rng)
        else:
            new_X, new_y = X, y
        ret.append(_build(new_X, new_y, rng, depth = depth, k=k))
    return ret

def random_forest_predict(X, f):
    thresh = len(f)//2
    curr = np.zeros(X.shape[0])
    for tree in f:
        curr += predict(X, tree)
    return (curr > thresh)

In [ ]:
from sklearn.datasets import load_breast_cancer, make_classification
from sklearn.model_selection import train_test_split
from sklearn.tree import export_text
from sklearn.ensemble import RandomForestClassifier
from collections import deque

def _display(curr: DecisionNode | LeafNode | None):
    if type(curr) is DecisionNode:
        print(f"col: {curr.col}, threshold: {curr.threshold:.2f}", end = " | ")
    elif type(curr) is LeafNode:
        print(f"res: {curr.res}", end = " | ")
    else:
        print(f"res : {None}")
        
def _display_graph(tree):
    q, newq = deque([tree]), deque([])
    while q or newq:
        while q:
            curr = q.pop()
            _display(curr)
            if not curr or type(curr) is not DecisionNode: continue
            for item in [curr.leftNode, curr.rightNode]:
                newq.appendleft(item)
        q, newq = newq, deque()
        print("", end ="\n")

In [5]:
X,y = make_classification(n_samples = 200, n_features = 4, random_state = 0)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.25, random_state = 0)

clf = RandomForestClassifier(criterion= "gini", max_depth=6, random_state=2, n_estimators=3)
clf.fit(X_train, y_train)
node = create_random_forest(X_train, y_train, depth = 6,num_trees = 3, rng = np.random.default_rng(2) )

print(export_text(clf.estimators_[0]))
_display_graph(node[0])

sk_preds = clf.predict(X_test)
my_preds = random_forest_predict(X_test, node)
print(sk_preds)
print([int(res) for res in my_preds])



|--- feature_3 <= -0.13
|   |--- feature_0 <= 0.16
|   |   |--- class: 0.0
|   |--- feature_0 >  0.16
|   |   |--- class: 1.0
|--- feature_3 >  -0.13
|   |--- feature_0 <= 1.11
|   |   |--- feature_3 <= 0.11
|   |   |   |--- feature_2 <= -0.20
|   |   |   |   |--- feature_2 <= -0.42
|   |   |   |   |   |--- class: 0.0
|   |   |   |   |--- feature_2 >  -0.42
|   |   |   |   |   |--- class: 1.0
|   |   |   |--- feature_2 >  -0.20
|   |   |   |   |--- class: 0.0
|   |   |--- feature_3 >  0.11
|   |   |   |--- class: 0.0
|   |--- feature_0 >  1.11
|   |   |--- class: 1.0

col: 0, threshold: 0.36 | 
col: 2, threshold: 0.01 | col: 0, threshold: 0.05 | 
col: 2, threshold: 0.07 | res: 1 | col: 0, threshold: 0.06 | res: 0 | 
res: 1 | res: 0 | col: 1, threshold: -0.79 | res: 1 | 
col: 3, threshold: 0.03 | res: 1 | 
col: 3, threshold: 0.20 | res: 0 | 
res: 0 | res: 1 | 
[1 0 0 1 0 1 1 1 1 0 0 0 0 0 1 1 1 0 0 0 1 0 1 1 0 1 0 1 1 0 1 0 1 1 0 0 1
 0 0 1 1 1 0 1 1 0 0 0 0 1]
[1, 0, 0, 1, 0, 1, 1, 1, 

In [ ]:
from sklearn.metrics import classification_report
print(classification_report(y_test, sk_preds))
print(classification_report(y_test, my_preds))
np.sum(np.array(my_preds) != sk_preds)

              precision    recall  f1-score   support

           0       0.96      0.92      0.94        26
           1       0.92      0.96      0.94        24

    accuracy                           0.94        50
   macro avg       0.94      0.94      0.94        50
weighted avg       0.94      0.94      0.94        50

              precision    recall  f1-score   support

           0       0.96      0.88      0.92        26
           1       0.88      0.96      0.92        24

    accuracy                           0.92        50
   macro avg       0.92      0.92      0.92        50
weighted avg       0.92      0.92      0.92        50



np.int64(1)

In [ ]:
Xb, yb = load_breast_cancer(return_X_y=True)
Xb_train, Xb_test, yb_train, yb_test = train_test_split(Xb, yb, test_size=0.2, random_state=0, stratify=yb)

num_trees = [1, 10, 50, 100]
k = [i for i in range(int((X.shape[1]) ** 1/2), X.shape[1])]
bootstrap = [0, 1]

clf = RandomForestClassifier(criterion= "gini", max_depth=6, random_state=2, n_estimators=3)
clf.fit(Xb_train, yb_train)
node = create_random_forest(Xb_train, yb_train, depth = 6, k = 5, num_trees = 3, rng = np.random.default_rng(2) )

# print(export_text(clf.estimators_[0]))
# _display_graph(node[0])

sk_preds = clf.predict(Xb_test)
my_preds = random_forest_predict(Xb_test, node)
# print(sk_preds)
# print([int(res) for res in my_preds])

from sklearn.metrics import classification_report
print(classification_report(yb_test, sk_preds))
print(classification_report(yb_test, my_preds))
np.sum(np.array(my_preds) != sk_preds)

|--- feature_7 <= 0.07
|   |--- feature_20 <= 16.80
|   |   |--- feature_21 <= 34.25
|   |   |   |--- feature_10 <= 0.63
|   |   |   |   |--- feature_7 <= 0.05
|   |   |   |   |   |--- class: 1.0
|   |   |   |   |--- feature_7 >  0.05
|   |   |   |   |   |--- feature_24 <= 0.15
|   |   |   |   |   |   |--- class: 1.0
|   |   |   |   |   |--- feature_24 >  0.15
|   |   |   |   |   |   |--- class: 0.0
|   |   |   |--- feature_10 >  0.63
|   |   |   |   |--- feature_7 <= 0.02
|   |   |   |   |   |--- class: 1.0
|   |   |   |   |--- feature_7 >  0.02
|   |   |   |   |   |--- class: 0.0
|   |   |--- feature_21 >  34.25
|   |   |   |--- feature_27 <= 0.11
|   |   |   |   |--- class: 1.0
|   |   |   |--- feature_27 >  0.11
|   |   |   |   |--- class: 0.0
|   |--- feature_20 >  16.80
|   |   |--- feature_12 <= 1.25
|   |   |   |--- class: 1.0
|   |   |--- feature_12 >  1.25
|   |   |   |--- feature_26 <= 0.20
|   |   |   |   |--- feature_22 <= 127.65
|   |   |   |   |   |--- class: 1.0
|   |  